# Fako Online - Styled Scene Server (Kaggle)

**AnimateDiff + ControlNet + SD 1.5**

Generates consistent character scenes with style transfer.

### Instructions
1. Enable **GPU T4 x2** (Settings > Accelerator)
2. Add dataset `kingtechie/fako-ai-models` (from primary account)
3. Run all cells in order
4. The server will start on port 8002

In [ ]:
# Cell 1: Setup paths
import os

KAGGLE_DATASET = "/kaggle/input/fako-ai-models"
WORKING_DIR = "/kaggle/working/outputs"
os.makedirs(WORKING_DIR, exist_ok=True)

SD15_DIR = f"{KAGGLE_DATASET}/SD1.5_Base"
ANIMEDIFF_DIR = f"{KAGGLE_DATASET}/AnimateDiff"

print(f"Dataset: {KAGGLE_DATASET}")
print(f"Working dir: {WORKING_DIR}")

In [ ]:
# Cell 2: Install dependencies
!pip install -q diffusers transformers accelerate
!pip install -q controlnet-aux
!pip install -q fastapi uvicorn python-multipart
!pip install -q imageio[ffmpeg]
print("Dependencies installed!")

In [ ]:
# Cell 3: Import libraries
import torch
import numpy as np
from PIL import Image
from diffusers import AnimateDiffPipeline, ControlNetModel
from diffusers.utils import export_to_video
import io
import base64

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Cell 4: Load AnimateDiff + ControlNet pipeline
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11f1p_sd15_depth",
    torch_dtype=torch.float16
)

pipe = AnimateDiffPipeline.from_pretrained(
    SD15_DIR,
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe.to("cuda")

print("AnimateDiff + ControlNet loaded!")

In [ ]:
# Cell 5: Styled scene generation function
def generate_styled_scene(image_path, prompt, style_prompt, duration=5, num_frames=16):
    init_image = Image.open(image_path).resize((512, 512))
    video_frames = pipe(
        prompt=f"{prompt}, {style_prompt}",
        image=init_image,
        num_frames=num_frames,
        num_inference_steps=20,
        guidance_scale=7.5,
        height=512,
        width=512
    ).frames[0]
    output_path = f"{WORKING_DIR}/styled_scene.mp4"
    export_to_video(video_frames, output_path, fps=16)
    return output_path

print("Styled scene generation function defined!")

In [ ]:
# Cell 6: Image generation function (SD 1.5 txt2img)
def generate_image(prompt, width=512, height=512):
    from diffusers import StableDiffusionPipeline

    sd_pipe = StableDiffusionPipeline.from_pretrained(
        SD15_DIR,
        torch_dtype=torch.float16,
        safety_checker=None
    )
    sd_pipe.to("cuda")

    image = sd_pipe(
        prompt=prompt,
        width=width,
        height=height,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]

    output_path = f"{WORKING_DIR}/generated_image.png"
    image.save(output_path)

    del sd_pipe
    torch.cuda.empty_cache()

    return output_path

print("Image generation function defined!")

In [ ]:
# Cell 7: FastAPI server
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title="Fako Online - Styled Scene API")

@app.get("/health")
async def health():
    return {"status": "ok", "models": ["animatediff", "controlnet", "sd1.5"]}

@app.post("/generate-image")
async def api_generate_image(text_prompt: str = Form(...), width: int = Form(512), height: int = Form(512)):
    try:
        image_path = generate_image(text_prompt, width, height)
        return FileResponse(image_path, media_type="image/png")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.post("/generate-styled")
async def api_generate_styled(
    image: UploadFile = File(...),
    text_prompt: str = Form(...),
    style_prompt: str = Form(...),
    duration: int = Form(5)
):
    try:
        image_path = f"{WORKING_DIR}/{image.filename}"
        with open(image_path, "wb") as f:
            f.write(await image.read())
        video_path = generate_styled_scene(image_path, text_prompt, style_prompt, duration)
        return FileResponse(video_path, media_type="video/mp4")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

print("FastAPI server defined!")

In [ ]:
# Cell 8: Start server
print("Starting server on port 8002...")
uvicorn.run(app, host="0.0.0.0", port=8002)